In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Import sckit-learn tools
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler #to scale the data
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score,
    precision_score,
    f1_score
)
import warnings
warnings.filterwarnings('ignore')

#Visualization setting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15,7)

print("Libraries imported")

Libraries imported


StandarScale, permite normalizar mis caracteristicas.


In [2]:
print("="*40, "Step 1: Data Loading", "="*40)

DATA_PATH = '../../data/mixed_population_dataset_160_households.csv'
print(f"Reading data from: {DATA_PATH}...")

df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("\nThe data was successfully downloaded!")
print(f"Dataset Shape: {df.shape}")
print(f"Time period: from {df['timestamp'].min()} to {df['timestamp'].max()}")

======================================== Step 1: Data Loading ========================================
Reading data from: ../../data/mixed_population_dataset_160_households.csv...

The data was successfully downloaded!
Dataset Shape: (1382400, 4)
Time period: from 2025-11-03 00:00:00 to 2026-01-31 23:45:00


NEW

In [3]:
print("\n"+"="*40, "Step 2: Feature Engineering", "="*40)

#Basic temporary features
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print("Basic Features created!")

# --- 2.2: Características de Ventana Deslizante (Rolling Window) ---
# IDEA: Una fuga es un consumo BAJO pero CONSTANTE. Un pico de consumo es ALTO y CORTO.
# Las ventanas deslizantes ayudan al modelo a ver esta diferencia.

# ¡CRUCIAL! Debemos ordenar los datos por hogar y fecha para que las ventanas
# no se calculen sobre datos de hogares diferentes.
df = df.sort_values(['household_id', 'timestamp']).reset_index(drop=True)

# Definimos los tamaños de las ventanas (en nº de periodos de 15 min).
# 4 periodos = 1 hora, 8 = 2 horas, 24 = 6 horas.
window_sizes = [4, 8, 24]

grouped = df.groupby('household_id')['consumption_l']

for window in window_sizes:
    # .groupby('household_id') es clave. Aplica el .rolling() a cada hogar por separado.
    # .transform() es un truco de pandas que devuelve los resultados con el mismo índice que el df original.
    # Media y Desviación Estándar
    rolling_mean = grouped.transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    rolling_std = grouped.transform(lambda x: x.rolling(window=window, min_periods=1).std())

    df[f'rolling_cv_{window}'] = rolling_std / (rolling_mean + 1e-6)
    
    # También podemos mantener la media y la std si queremos
    df[f'rolling_mean_{window}'] = rolling_mean
    df[f'rolling_std_{window}'] = rolling_std

# Rellenar los valores nulos (NaN) que se crean al principio de cada ventana
df.fillna(0, inplace=True)

print("✓ Window feature created.")
print("\nDataFrame with all the new features:")
print(df.head(10))


======================================== Step 2: Feature Engineering ========================================
Basic Features created!
✓ Window feature created.

DataFrame with all the new features:
            timestamp  consumption_l  is_leak  \
0 2025-11-03 00:00:00       0.000000        0   
1 2025-11-03 00:15:00      32.850753        0   
2 2025-11-03 00:30:00       0.000000        0   
3 2025-11-03 00:45:00       0.000000        0   
4 2025-11-03 01:00:00       0.000000        0   
5 2025-11-03 01:15:00       0.000000        0   
6 2025-11-03 01:30:00       0.000000        0   
7 2025-11-03 01:45:00       0.000000        0   
8 2025-11-03 02:00:00       0.000000        0   
9 2025-11-03 02:15:00       0.000000        0   

                      household_id  hour  day_of_week  is_weekend  \
0  household_apartment_family_1041     0            0           0   
1  household_apartment_family_1041     0            0           0   
2  household_apartment_family_1041     0            0 

Explicación:
sort_values(...): Este paso es fundamental. Antes de calcular una media móvil, los datos deben estar en orden cronológico para cada hogar.
rolling_mean: Para cada punto en el tiempo, calcula la media del consumo de los window puntos anteriores (ej. la última hora). Si hay una fuga, esta media será constante y mayor que cero. Si no, fluctuará.
rolling_std: Calcula la desviación estándar en la misma ventana. Si hay una fuga, la desviación será muy baja (el consumo es constante). Si hay consumo normal (picos y valles), será alta. Esta es una pista potentísima para el modelo.
.groupby('household_id').transform(...): Esta es la forma profesional de aplicar funciones de ventana en Pandas cuando tienes múltiples grupos (en nuestro caso, múltiples hogares). Asegura que el cálculo se reinicia para cada nuevo household_id.
fillna(0): Las primeras filas de cada hogar no tienen suficientes datos anteriores para calcular la ventana completa, por lo que Pandas pone NaN (Not a Number). Los rellenamos con 0.

In [4]:
print("\n"+"="*40, "Step 3: Data Division (Chronologically)", "="*40)

# Para series temporales, NUNCA debemos mezclar el futuro con el pasado.
# Entrenamos con los datos más antiguos y probamos con los más recientes.
# Esto simula un escenario real: entrenas un modelo con datos históricos
# y lo usas para predecir sobre datos nuevos que van llegando.

# Lista final de características que usaremos

feature_cols = [
    'consumption_l',
    'hour',
    'is_weekend',
    'rolling_mean_4',
    'rolling_std_4',
    'rolling_cv_4',
    'rolling_mean_8',
    'rolling_std_8',
    'rolling_cv_8',
    'rolling_mean_24',
    'rolling_std_24',
    'rolling_cv_24'
]

target = 'is_leak'

#sorting the df by date
df = df.sort_values('timestamp')
#Corto en el 80% de los datos 
split_point = int(len(df) *0.8)

#Creating the training sets and test set
train_df = df.iloc[:split_point]
test_df = df.iloc[split_point:]

X_train = train_df[feature_cols]
y_train = train_df[target]
X_test = test_df[feature_cols]
y_test = test_df[target]

print("Chronological division completed!")
print(f"Training: from {train_df['timestamp'].min()} to {train_df['timestamp'].max()}")
print(f"Test:        from {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")

print("\n"+"="*40, "Step 4: Data Scalation", "="*40)

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

print("Data Scaled successfully")



======================================== Step 3: Data Division (Chronologically) ========================================
Chronological division completed!
Training: from 2025-11-03 00:00:00 to 2026-01-13 23:45:00
Test:        from 2026-01-14 00:00:00 to 2026-01-31 23:45:00

======================================== Step 4: Data Scalation ========================================
Data Scaled successfully


In [8]:
from sklearn.neighbors import LocalOutlierFactor

print("\n"+"="*40, "Step 5: Looking for the best pollution parameter Using Local Outlier Factor instad of Isolation Forest", "="*40)

contamination_values = [0.002, 0.005, 0.01, 0.015, 0.02, 0.03]
results_lof = []

print("Testing different contamination values")

for contam in contamination_values:
    print(f"\n--- Training with contamination = {contam}---")
    #model = IsolationForest(
    #    n_estimators = 100,
    #    contamination = contam,
    #    random_state = 42,
    #    n_jobs = -1
    #)

    lof_model = LocalOutlierFactor(
        n_neighbors = 20,
        contamination = contam,
        novelty = True,
        n_jobs = -1
    )

    #train with scaled data
    lof_model.fit(x_train_scaled)

    #predict over test 
    y_pred = lof_model.predict(x_test_scaled)
    y_pred_mapped = np.where(y_pred == -1, 1, 0)

    #Get metrics
    recall = recall_score(y_test, y_pred_mapped)
    precision = precision_score(y_test, y_pred_mapped, zero_division=0)
    f1 = f1_score(y_test, y_pred_mapped, zero_division=0)

    results_lof.append({'contamination': contam, 'recall': recall, 'precision': precision, 'F1_score': f1})

    print(f"Recall: {recall: .4f}")
    print(f"Precision: {precision: .4f}")
    print(f"  F1-Score: {f1:.4f}")


#Add the results into a DF
results_lof_df = pd.DataFrame(results_lof)

print("\n--- Research Results ---")
print(results_lof_df)


======================================== Step 5: Looking for the best pollution parameter Using Local Outlier Factor instad of Isolation Forest ========================================
Testing different contamination values

--- Training with contamination = 0.002---
Recall:  0.0047
Precision:  0.0038
  F1-Score: 0.0042

--- Training with contamination = 0.005---
Recall:  0.0312
Precision:  0.0099
  F1-Score: 0.0150

--- Training with contamination = 0.01---
Recall:  0.0641
Precision:  0.0108
  F1-Score: 0.0184

--- Training with contamination = 0.015---
Recall:  0.1062
Precision:  0.0119
  F1-Score: 0.0215

--- Training with contamination = 0.02---
Recall:  0.1375
Precision:  0.0117
  F1-Score: 0.0215

--- Training with contamination = 0.03---
Recall:  0.1812
Precision:  0.0106
  F1-Score: 0.0201

--- Research Results ---
   contamination    recall  precision  F1_score
0          0.002  0.004687   0.003802  0.004199
1          0.005  0.031250   0.009906  0.015043
2          0.010  0.